generate all possible call types to understand how to pic care eligible calls.

In [10]:
import duckdb

con = duckdb.connect()
final_file = '../data_processed/calls_2025_deduplicated_final.parquet'
output_csv = '../data_processed/final_call_types_2025_full.csv'
#check how many nulls
# This query counts total rows, non-null values, and null values
check_nulls_query = f"""
SELECT 
    COUNT(*) AS total_rows,
    -- Count non-null values
    COUNT(final_call_type) AS non_null_final_count,
    COUNT(initial_call_type) AS non_null_initial_count,
    -- Count null values specifically
    SUM(CASE WHEN final_call_type IS NULL THEN 1 ELSE 0 END) AS null_final_count,
    SUM(CASE WHEN initial_call_type IS NULL THEN 1 ELSE 0 END) AS null_initial_count
FROM '{final_file}'
"""

df_nulls = con.execute(check_nulls_query).df()
print(df_nulls)

query = f"""
SELECT 
    final_call_type,
    COUNT(*) as frequency,
    -- We'll also see the most common initial type for each final type 
    -- to understand the "dispatch evolution"
    MODE(initial_call_type) as most_common_initial_type
FROM '{final_file}'
WHERE final_call_type IS NOT NULL
GROUP BY 1
ORDER BY 2 DESC
"""

df_all_types = con.execute(query).df()

# 3. Save to CSV
df_all_types.to_csv(output_csv, index=False)

print(f"✅ Full list exported to: {output_csv}")
print(f"Total Unique Call Types Found: {len(df_all_types)}")

# 4. Show the top 20 here for immediate discussion
print("\n--- TOP 20 CALL TYPES ---")
print(df_all_types.head(20))

con.close()

   total_rows  non_null_final_count  non_null_initial_count  null_final_count  \
0      324386                324386                  324386               0.0   

   null_initial_count  
0                 0.0  
✅ Full list exported to: ../data_processed/final_call_types_2025_full.csv
Total Unique Call Types Found: 352

--- TOP 20 CALL TYPES ---
                                  final_call_type  frequency  \
0          SUSPICIOUS CIRCUM. - SUSPICIOUS PERSON      29916   
1                             DISTURBANCE - OTHER      29186   
2           ASSIST PUBLIC - OTHER (NON-SPECIFIED)      23636   
3   TRAFFIC - PARKING VIOL (EXCEPT ABANDONED CAR)      18508   
4                      TRAFFIC - MOVING VIOLATION      14476   
5                        DIRECTED PATROL ACTIVITY      13405   
6                      CRISIS COMPLAINT - GENERAL      12851   
7               PREMISE CHECKS - CRIME PREVENTION      12019   
8                              PROWLER - TRESPASS      10744   
9            

In [11]:
import duckdb
import pandas as pd

# 1. Setup paths
con = duckdb.connect()
input_file = '../data_processed/calls_2025_deduplicated_final.parquet'
output_csv = '../data_processed/response_category_call_type_distribution.csv'

print(f"📊 Analyzing {input_file}...")

# 2. Get high-level stats for Response Categories
stats_query = f"""
SELECT 
    cad_event_response_category, 
    COUNT(*) as frequency,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM '{input_file}'), 2) as percentage
FROM '{input_file}'
GROUP BY 1
ORDER BY 2 DESC
"""
df_stats = con.execute(stats_query).df()

# 3. Get the FULL distribution (all call types for every category)
# We sort by category and then the most frequent call types
full_dist_query = f"""
SELECT 
    cad_event_response_category,
    final_call_type,
    COUNT(*) as incident_count
FROM '{input_file}'
GROUP BY 1, 2
ORDER BY 1, 3 DESC
"""
df_full_dist = con.execute(full_dist_query).df()

# 4. Save to CSV
df_full_dist.to_csv(output_csv, index=False)

print("\n--- 1. OVERALL RESPONSE STATS ---")
print(df_stats)

print(f"\n✅ 2. FULL DISTRIBUTION EXPORTED")
print(f"File saved to: {output_csv}")
print(f"Total unique combinations found: {len(df_full_dist)}")

# 5. Quick Preview: Top 3 types for each category
print("\n--- 3. PREVIEW: TOP 3 TYPES PER CATEGORY ---")
preview = df_full_dist.groupby('cad_event_response_category').head(3)
print(preview)

con.close()

📊 Analyzing ../data_processed/calls_2025_deduplicated_final.parquet...

--- 1. OVERALL RESPONSE STATS ---
  cad_event_response_category  frequency  percentage
0                         SPD     317556       97.89
1                        CARE       3847        1.19
2        SPD/CARE Co-Response       2983        0.92

✅ 2. FULL DISTRIBUTION EXPORTED
File saved to: ../data_processed/response_category_call_type_distribution.csv
Total unique combinations found: 489

--- 3. PREVIEW: TOP 3 TYPES PER CATEGORY ---
    cad_event_response_category                               final_call_type  \
0                          CARE                 CCR ONLY - COMMUNITY PRESENCE   
1                          CARE                  CCR ONLY - ASSIST THE PUBLIC   
2                          CARE  PREMISE CHECK, OFFICER INITIATED ONVIEW ONLY   
32                          SPD        SUSPICIOUS CIRCUM. - SUSPICIOUS PERSON   
33                          SPD                           DISTURBANCE - OTHER   
34

In [3]:
import duckdb

con = duckdb.connect()
input_file = '../data_processed/calls_2025_deduplicated_final.parquet'
output_tiered = '../data_processed/calls_2025_tiered_final_Re.parquet'

# regex patterns based on your tiered list
#tier_2_regex = 'CRISIS|WELFARE|SUICIDE|MENTAL|DOWN PERSON|OVERDOSE|CCR ONLY|MISSING|FOUND PERSON|CASUALTY'
#tier_1_regex = 'DISTURBANCE|ASSIST PUBLIC|INTOX|NUISANCE|HARASSMENT|THREATS|JUVENILE|RUNAWAY'

#refined the list
tier_2_regex = 'CRISIS|WELFARE|SUICIDE|MENTAL|DOWN PERSON|OVERDOSE|CCR ONLY|MISSING|FOUND PERSON|CASUALTY|DETOX|DOA|CHILD|ELDER'

# TIER 1: Social/Nuisance/Non-Violent Disorder
# Includes disturbances WITHOUT assaults and public assistance
tier_1_regex = 'DISTURBANCE|ASSIST PUBLIC|INTOX|NUISANCE|HARASSMENT|THREATS|JUVENILE|RUNAWAY|TRESPASS|PANHANDLING|ENCAMPMENT|URINATING|FIGHT|ANIMAL|SUSPICIOUS'
# The "Police Only" exclusion keywords (To prevent mis-tiering violence)
police_only_regex = 'ASLT|WEAPON|GUN|SHOTS|ROBBERY|BURGLARY|THEFT|HOMICIDE|RAPE|STAB|DV'

tiered_query = f"""
COPY (
    SELECT *,
        CASE 
            -- Priority 1: If it involves violence/crime, it's Tier 0 regardless of other keywords
            WHEN regexp_matches(upper(final_call_type), '{police_only_regex}') THEN 'Tier 0 - Traditional SPD'
            
            -- Priority 2: Clear behavioral/health vulnerability
            WHEN regexp_matches(upper(final_call_type), '{tier_2_regex}') THEN 'Tier 2 - Clearly CARE'
            
            -- Priority 3: Social/Nuisance (only if it didn't match the 'Police Only' check above)
            WHEN regexp_matches(upper(final_call_type), '{tier_1_regex}') THEN 'Tier 1 - Potential CARE'
            
            -- Priority 4: Everything else
            ELSE 'Tier 0 - Traditional SPD'
        END AS care_tier
    FROM '{input_file}'
) TO '{output_tiered}' (FORMAT PARQUET);
"""

print("🏗️ Categorizing 2025 incidents into Tiers 0, 1, and 2...")
con.execute(tiered_query)

# Final stats to see the distribution
stats = con.execute(f"SELECT care_tier, COUNT(*) as count, ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM '{output_tiered}'), 1) as pct FROM '{output_tiered}' GROUP BY 1 ORDER BY 1").df()

audit_query = f"""
WITH Classified AS (
    SELECT 
        final_call_type,
        COUNT(*) as freq,
        CASE 
            WHEN regexp_matches(upper(final_call_type), '{tier_2_regex}') THEN 'Tier 2'
            WHEN regexp_matches(upper(final_call_type), '{tier_1_regex}') THEN 'Tier 1'
            ELSE 'Tier 0'
        END AS assigned_tier
    FROM '{input_file}'
    GROUP BY 1
)
SELECT * FROM Classified ORDER BY assigned_tier DESC, freq DESC
"""

df_audit = con.execute(audit_query).df()

con.close()

print("\n--- TIERED RESPONSE BREAKDOWN ---")
print(stats)
# Let's see the results of the negative lookahead
print("--- TIER 1 SAMPLES (Successful Filtering) ---")
print(df_audit[df_audit['assigned_tier'] == 'Tier 1'].head(50))

print("\n--- TIER 0 SAMPLES (The Police Bucket) ---")
print(df_audit[df_audit['assigned_tier'] == 'Tier 0'].head(50))

🏗️ Categorizing 2025 incidents into Tiers 0, 1, and 2...

--- TIERED RESPONSE BREAKDOWN ---
                  care_tier   count   pct
0  Tier 0 - Traditional SPD  177837  54.8
1   Tier 1 - Potential CARE  125682  38.7
2     Tier 2 - Clearly CARE   20867   6.4
--- TIER 1 SAMPLES (Successful Filtering) ---
                                   final_call_type   freq assigned_tier
34          SUSPICIOUS CIRCUM. - SUSPICIOUS PERSON  29916        Tier 1
35                             DISTURBANCE - OTHER  29186        Tier 1
36           ASSIST PUBLIC - OTHER (NON-SPECIFIED)  23636        Tier 1
37                              PROWLER - TRESPASS  10744        Tier 1
38         SUSPICIOUS CIRCUM. - SUSPICIOUS VEHICLE   8120        Tier 1
39                             DISTURBANCE - NOISE   5076        Tier 1
40         DV - ARGUMENTS, DISTURBANCE (NO ARREST)   5050        Tier 1
41                  MISCHIEF OR NUISANCE - GENERAL   3800        Tier 1
42                  ASSAULTS - HARASSMENT, THR

Investigate movement from tier-1 to tier2:

In [6]:
import duckdb
import pandas as pd

con = duckdb.connect()
input_file = '../data_processed/calls_2025_deduplicated_final.parquet'

# Reuse the same robust regex definitions
tier_2_regex = 'CRISIS|WELFARE|SUICIDE|MENTAL|DOWN PERSON|OVERDOSE|CCR ONLY|MISSING|FOUND PERSON|CASUALTY|DETOX|DOA|CHILD|ELDER'
tier_1_regex = 'DISTURBANCE|ASSIST PUBLIC|INTOX|NUISANCE|HARASSMENT|THREATS|JUVENILE|RUNAWAY|TRESPASS|PANHANDLING|ENCAMPMENT|URINATING|FIGHT|ANIMAL|SUSPICIOUS'
police_only_regex = 'ASLT|WEAPON|GUN|SHOTS|ROBBERY|BURGLARY|THEFT|HOMICIDE|RAPE|STAB|DV'

progression_query = f"""
WITH Tiers AS (
    SELECT 
        -- Classify the INITIAL Call Type
        CASE 
            WHEN regexp_matches(upper(initial_call_type), '{police_only_regex}') THEN 'Tier 0 - Police'
            WHEN regexp_matches(upper(initial_call_type), '{tier_2_regex}') THEN 'Tier 2 - CARE'
            WHEN regexp_matches(upper(initial_call_type), '{tier_1_regex}') THEN 'Tier 1 - Potential'
            ELSE 'Tier 0 - Police'
        END AS initial_tier,

        -- Classify the FINAL Call Type
        CASE 
            WHEN regexp_matches(upper(final_call_type), '{police_only_regex}') THEN 'Tier 0 - Police'
            WHEN regexp_matches(upper(final_call_type), '{tier_2_regex}') THEN 'Tier 2 - CARE'
            WHEN regexp_matches(upper(final_call_type), '{tier_1_regex}') THEN 'Tier 1 - Potential'
            ELSE 'Tier 0 - Police'
        END AS final_tier
    FROM '{input_file}'
)
SELECT 
    initial_tier,
    final_tier,
    COUNT(*) as count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY initial_tier), 1) as pct_of_initial_tier
FROM Tiers
WHERE initial_tier = 'Tier 1 - Potential' -- We only care about the drift from Tier 1
GROUP BY 1, 2
ORDER BY 2
"""

df_progression = con.execute(progression_query).df()
con.close()

print("--- TIER 1 TRAJECTORY ANALYSIS ---")
print(df_progression)

--- TIER 1 TRAJECTORY ANALYSIS ---
         initial_tier          final_tier  count  pct_of_initial_tier
0  Tier 1 - Potential     Tier 0 - Police   7675                 10.6
1  Tier 1 - Potential  Tier 1 - Potential  62248                 85.9
2  Tier 1 - Potential       Tier 2 - CARE   2571                  3.5


In [8]:
import duckdb
import pandas as pd

con = duckdb.connect()
tiered_file = '../data_processed/calls_2025_tiered_final_Re.parquet'

# We look for "Onview" or "Officer" in the initial description to find self-initiated events
# We also check initial call types for weapons
stress_test_query = f"""
SELECT 
    care_tier,
    COUNT(*) as total_calls,
    
    -- 1. How many were Officer Initiated? (Likely not dispatchable to CARE)
    SUM(CASE WHEN initial_call_type LIKE '%ONVIEW%' 
               OR initial_call_type LIKE '%OFFICER%' 
               OR initial_call_type LIKE '%PREMISE%' THEN 1 ELSE 0 END) as officer_initiated_count,
               
    -- 2. How many started as Weapons/Violence? (Safety Risk)
    SUM(CASE WHEN (initial_call_type LIKE '%WEAPON%' 
               OR initial_call_type LIKE '%GUN%' 
               OR initial_call_type LIKE '%ASLT%') 
               AND care_tier != 'Tier 0 - Traditional SPD' THEN 1 ELSE 0 END) as hidden_violence_count

FROM '{tiered_file}'
GROUP BY 1
ORDER BY 1
"""

df_stress = con.execute(stress_test_query).df()
con.close()

print("--- DATA STRESS TEST ---")
print(df_stress)

--- DATA STRESS TEST ---
                  care_tier  total_calls  officer_initiated_count  \
0  Tier 0 - Traditional SPD       177837                  30981.0   
1   Tier 1 - Potential CARE       125682                   4155.0   
2     Tier 2 - Clearly CARE        20867                   2189.0   

   hidden_violence_count  
0                    0.0  
1                 7173.0  
2                  643.0  


Random sanity check on regex op. can ignore.

In [13]:
import duckdb
import pandas as pd
#random check
con = duckdb.connect()
tiered_file = '../data_processed/calls_2025_tiered_final_Re.parquet'

# We look for "Onview" or "Officer" in the initial description to find self-initiated events
# We also check initial call types for weapons
stress_test_query = f"""
SELECT DISTINCT initial_call_type
FROM '{tiered_file}'
WHERE regexp_matches(
    LOWER(initial_call_type),
    '(shot at|shots fired|armed)'
)
LIMIT 50;

"""

df_stress = con.execute(stress_test_query).df()


stress_test_query2 = f"""
SELECT DISTINCT initial_call_type
FROM '{tiered_file}'
WHERE regexp_matches( LOWER(initial_call_type),'(found gun|shell casings)')
LIMIT 50;

"""

df_stress2 = con.execute(stress_test_query2).df()

stress_test_query3 = f"""
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN regexp_matches(LOWER(initial_call_type), 'gun') THEN 1 ELSE 0 END) AS mentions_gun
FROM '{tiered_file}';

"""

df_stress3 = con.execute(stress_test_query3).df()
con.close()

print("--- DATA STRESS TEST ---")
print(df_stress)
print("--- DATA STRESS TEST 2---")
print(df_stress2)
print("--- DATA STRESS TEST 3---")
print(df_stress3)


--- DATA STRESS TEST ---
                          initial_call_type
0  ASLT - REPORT FOR PERSON SHOT OR SHOT AT
1             ASLT - PERSON SHOT OR SHOT AT
--- DATA STRESS TEST 2---
                    initial_call_type
0  PROPERTY - FOUND GUN, SHELLCASINGS
--- DATA STRESS TEST 3---
    total  mentions_gun
0  324386        1939.0


In [14]:
import duckdb
import pandas as pd

con = duckdb.connect()
tiered_file = '../data_processed/calls_2025_tiered_final_Re.parquet'

# We look for "Onview" or "Officer" in the initial description to find self-initiated events
# We also check initial call types for weapons
stress_test_query = f"""
WITH base AS (
    SELECT
        care_tier,
        LOWER(initial_call_type) AS call_text
    FROM '{tiered_file}'
),

classified AS (
    SELECT
        care_tier,
        call_text,

        /* ---------------------------------
           OFFICER-INITIATED (STRICT)
           --------------------------------- */
        CASE
            WHEN regexp_matches(
                call_text,
                '(onview|officer initiated|traffic stop)'
            )
            THEN 1 ELSE 0
        END AS officer_initiated_flag,

        /* ---------------------------------
           SAFETY TIER (INITIAL INTAKE SIGNAL)
           --------------------------------- */
        CASE
            /* Tier 2: Active interpersonal threat */
            WHEN (
                regexp_matches(
                    call_text,
                    '(shot at|person shot|shots fired|drive by|armed suspect|armed person|stabbing|knife attack|bomb|explosion|firearm|gun|weapon|assault|aslt|fight)'
                )
                AND NOT regexp_matches(
                    call_text,
                    '(found gun|found weapon|recovered|secured|shell casings|property|alarm)'
                )
            )
            THEN 2

            /* Tier 1: Weapon present, no active threat */
            WHEN regexp_matches(
                call_text,
                '(found gun|found weapon|recovered weapon|secured weapon|shell casings|property.*gun|gun found)'
            )
            THEN 1

            /* Tier 0: No safety signal */
            ELSE 0
        END AS safety_tier

    FROM base
)

SELECT
    care_tier,
    COUNT(*) AS total_calls,

    /* Officer-initiated */
    SUM(officer_initiated_flag) AS officer_initiated_count,
    ROUND(100.0 * SUM(officer_initiated_flag) / COUNT(*), 1)
        AS pct_officer_initiated,

    /* Safety tiers */
    SUM(CASE WHEN safety_tier = 2 THEN 1 ELSE 0 END)
        AS active_threat_count,
    ROUND(100.0 * SUM(CASE WHEN safety_tier = 2 THEN 1 ELSE 0 END) / COUNT(*), 1)
        AS pct_active_threat,

    SUM(CASE WHEN safety_tier = 1 THEN 1 ELSE 0 END)
        AS weapon_present_no_threat_count,
    ROUND(100.0 * SUM(CASE WHEN safety_tier = 1 THEN 1 ELSE 0 END) / COUNT(*), 1)
        AS pct_weapon_present_no_threat

FROM classified
GROUP BY care_tier
ORDER BY care_tier
"""


df_stress = con.execute(stress_test_query).df()
con.close()

print("--- DATA STRESS TEST ---")
print(df_stress)

--- DATA STRESS TEST ---
                  care_tier  total_calls  officer_initiated_count  \
0  Tier 0 - Traditional SPD       177837                  30927.0   
1   Tier 1 - Potential CARE       125682                   4143.0   
2     Tier 2 - Clearly CARE        20867                   2188.0   

   pct_officer_initiated  active_threat_count  pct_active_threat  \
0                   17.4              11672.0                6.6   
1                    3.3               7576.0                6.0   
2                   10.5                697.0                3.3   

   weapon_present_no_threat_count  pct_weapon_present_no_threat  
0                           626.0                           0.4  
1                           132.0                           0.1  
2                             2.0                           0.0  


In [16]:
import duckdb

con = duckdb.connect()
input_file = '../data_processed/calls_2025_deduplicated_final.parquet'
output_tiered = '../data_processed/calls_2025_tiered_final_Re.parquet'

# regex patterns based on your tiered list
#tier_2_regex = 'CRISIS|WELFARE|SUICIDE|MENTAL|DOWN PERSON|OVERDOSE|CCR ONLY|MISSING|FOUND PERSON|CASUALTY'
#tier_1_regex = 'DISTURBANCE|ASSIST PUBLIC|INTOX|NUISANCE|HARASSMENT|THREATS|JUVENILE|RUNAWAY'

#refined the list
tier_2_regex = 'CRISIS|WELFARE|SUICIDE|MENTAL|DOWN PERSON|OVERDOSE|CCR ONLY|MISSING|FOUND PERSON|CASUALTY|DETOX|DOA|CHILD|ELDER'

# TIER 1: Social/Nuisance/Non-Violent Disorder
# Includes disturbances WITHOUT assaults and public assistance
tier_1_regex = 'DISTURBANCE|ASSIST PUBLIC|INTOX|NUISANCE|HARASSMENT|THREATS|JUVENILE|RUNAWAY|TRESPASS|PANHANDLING|ENCAMPMENT|URINATING|FIGHT|ANIMAL|SUSPICIOUS'
# The "Police Only" exclusion keywords (To prevent mis-tiering violence)
police_only_regex = 'ASLT|WEAPON|GUN|SHOTS|ROBBERY|BURGLARY|THEFT|HOMICIDE|RAPE|STAB|DV'

tiered_query = f"""
COPY (
    SELECT *,
        CASE 
            -- Priority 1: If it involves violence/crime, it's Tier 0 regardless of other keywords
            WHEN regexp_matches(upper(final_call_type), '{police_only_regex}') THEN 'Tier 0 - Traditional SPD'
            
            -- Priority 2: Clear behavioral/health vulnerability
            WHEN regexp_matches(upper(final_call_type), '{tier_2_regex}') THEN 'Tier 2 - Clearly CARE'
            
            -- Priority 3: Social/Nuisance (only if it didn't match the 'Police Only' check above)
            WHEN regexp_matches(upper(final_call_type), '{tier_1_regex}') THEN 'Tier 1 - Potential CARE'
            
            -- Priority 4: Everything else
            ELSE 'Tier 0 - Traditional SPD'
        END AS care_tier
    FROM '{input_file}'
) TO '{output_tiered}' (FORMAT PARQUET);
"""

print("🏗️ Categorizing 2025 incidents into Tiers 0, 1, and 2...")
con.execute(tiered_query)

# Final stats to see the distribution
stats = con.execute(f"SELECT cad_event_response_category, care_tier, COUNT(*) as count, ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM '{output_tiered}'), 1) as pct FROM '{output_tiered}' GROUP BY 1, 2 ORDER BY 1").df()

con.close()

print("\n--- TIERED RESPONSE BREAKDOWN ---")
print(stats)


🏗️ Categorizing 2025 incidents into Tiers 0, 1, and 2...

--- TIERED RESPONSE BREAKDOWN ---
  cad_event_response_category                 care_tier   count   pct
0                        CARE     Tier 2 - Clearly CARE    3087   1.0
1                        CARE  Tier 0 - Traditional SPD     709   0.2
2                        CARE   Tier 1 - Potential CARE      51   0.0
3                         SPD   Tier 1 - Potential CARE  124118  38.3
4                         SPD  Tier 0 - Traditional SPD  176658  54.5
5                         SPD     Tier 2 - Clearly CARE   16780   5.2
6        SPD/CARE Co-Response  Tier 0 - Traditional SPD     470   0.1
7        SPD/CARE Co-Response     Tier 2 - Clearly CARE    1000   0.3
8        SPD/CARE Co-Response   Tier 1 - Potential CARE    1513   0.5


Looking at CAD example that fall into responded by care but safety tier was level 0.

In [19]:
import duckdb

con = duckdb.connect()
raw_file = '../data_processed/calls_2025_full.parquet'

# Regex for Tier 0 (Police Only)
police_only_regex = 'ASLT|WEAPON|GUN|SHOTS|ROBBERY|BURGLARY|THEFT|HOMICIDE|RAPE|STAB|DV'

# 1. Find 2 unique CAD IDs that are 'CARE' category but 'Tier 0' final type
find_query = f"""
SELECT DISTINCT cad_event_number
FROM '{raw_file}'
WHERE cad_event_response_category = 'CARE'
  AND regexp_matches(upper(final_call_type), '{police_only_regex}')
LIMIT 2
"""
target_cads = con.execute(find_query).df()['cad_event_number'].tolist()

# 2. Inspect the Life Cycle of these CADs
if not target_cads:
    print("No matching mismatched CADs found.")
else:
    for cad_id in target_cads:
        print(f"\n🔍 --- Timeline for CAD: {cad_id} ---")
        inspect_query = f"""
        SELECT 
            cad_event_number,
            initial_call_type,
            final_call_type,
            cad_event_clearance_description,
            -- Check if CARE was actually at the scene
            CASE WHEN first_care_call_sign_at_scene_time IS NOT NULL THEN 'At Scene' ELSE 'Not at Scene' END as care_status,
            -- Check if SPD was also there (Co-Response hidden in CARE category)
            CASE WHEN first_spd_call_sign_at_scene_time IS NOT NULL THEN 'At Scene' ELSE 'Not at Scene' END as spd_status,
            count_of_officers
        FROM '{raw_file}'
        WHERE cad_event_number = {cad_id}
        """
        # Transpose (T) makes it easier to read a few rows of many columns
        print(con.execute(inspect_query).df().T)

con.close()


🔍 --- Timeline for CAD: 2025000254737 ---
                                                                                 0
cad_event_number                                                     2025000254737
initial_call_type                ASSIST PUBLIC - NO WELFARE CHK OR DV ORDER SER...
final_call_type                  ASSIST PUBLIC - NO WELFARE CHK OR DV ORDER SER...
cad_event_clearance_description          CCR only - RESOURCES OR SUPPLIES PROVIDED
care_status                                                               At Scene
spd_status                                                            Not at Scene
count_of_officers                                                                2

🔍 --- Timeline for CAD: 2025000258484 ---
                                                                                 0
cad_event_number                                                     2025000258484
initial_call_type                ASSIST PUBLIC - NO WELFARE CHK OR DV ORDER SER...
f

In [20]:
import pandas as pd

# Define the file path
file_path = '../data_processed/calls_2025_fresh_api.parquet'

# Police only regex (Tier 0)
police_only_regex = 'ASLT|WEAPON|GUN|SHOTS|ROBBERY|BURGLARY|THEFT|HOMICIDE|RAPE|STAB|DV'

# Use pandas to read the parquet. 
# Since pyarrow failed before, let's try if fastparquet is available or if pyarrow was a fluke.
# Alternatively, I'll try to use duckdb again, maybe the previous error was environment specific.

try:
    import duckdb
    con = duckdb.connect()
    
    # 1. Find the target CADs
    find_query = f"""
    SELECT DISTINCT cad_event_number
    FROM '{file_path}'
    WHERE cad_event_response_category = 'CARE'
      AND regexp_matches(upper(final_call_type), '{police_only_regex}')
    LIMIT 2
    """
    target_df = con.execute(find_query).df()
    target_cads = target_df['cad_event_number'].tolist()
    
    if target_cads:
        # 2. Check counts and get all rows
        for cad_id in target_cads:
            count_query = f"SELECT COUNT(*) FROM '{file_path}' WHERE cad_event_number = {cad_id}"
            count = con.execute(count_query).fetchone()[0]
            print(f"\nCAD ID: {cad_id} has {count} rows.")
            
            all_rows_query = f"SELECT * FROM '{file_path}' WHERE cad_event_number = {cad_id} ORDER BY cad_event_original_time_queued"
            df_all = con.execute(all_rows_query).df()
            print(f"Full data for {cad_id}:")
            print(df_all.to_string())
    else:
        print("No target CADs found.")
        
except Exception as e:
    print(f"An error occurred: {e}")


CAD ID: 2025000342982 has 2 rows.
Full data for 2025000342982:
   cad_event_number            cad_event_clearance_description call_type  priority                                   initial_call_type                                     final_call_type cad_event_original_time_queued cad_event_arrived_time dispatch_precinct dispatch_sector dispatch_beat dispatch_longitude dispatch_latitude dispatch_reporting_area cad_event_response_category call_sign_dispatch_id call_sign_dispatch_time first_care_call_sign_at_scene_time first_care_call_sign_dispatch_time first_co_response_call_sign_at_scene_time first_co_response_call_sign_dispatch_time first_spd_call_sign_at_scene_time first_spd_call_sign_dispatch_time last_care_call_sign_in_service_time last_co_response_call_sign_in_service_time last_spd_call_sign_in_service_time  care_call_sign_total_service_time_s_  co_response_call_sign_total_service_time_s_  spd_call_sign_total_service_time_s_  call_sign_total_service_time_s_  first_care_call_sign_d

In [2]:
import pandas as pd
import os

# Path to your tiered data
input_path = '../data_processed/calls_2025_tiered_final_Re.parquet'
output_csv = 'distinct_neighbourhoods_addresses_2025.csv'

if os.path.exists(input_path):
    df = pd.read_parquet(input_path)
    
    # 1. Get Distinct Neighborhoods and their Counts
    # value_counts() is the most efficient way to get both unique values and their frequency
    neighborhood_counts = df['dispatch_neighborhood'].value_counts().sort_index()
    
    print(f"Total Unique Dispatch Neighborhoods: {len(neighborhood_counts)}")
    print("\n--- Neighborhood Distribution (Call Counts) ---")
    print(neighborhood_counts)

    # 2. Extract unique neighborhood/address pairs for the "G2" mapping
    # This identifies the specific locations within each neighborhood
    pairs = df[['dispatch_neighborhood', 'dispatch_address']].drop_duplicates()
    
    # Save to CSV for external mapping/audit
    pairs.to_csv(output_csv, index=False)
    
    print(f"\nSuccessfully saved {len(pairs)} unique neighborhood/address pairs to: {output_csv}")
else:
    print(f"File not found: {input_path}")
    print("Tip: Check if the 'data_processed' folder is correctly located in the parent directory.")

Total Unique Dispatch Neighborhoods: 60

--- Neighborhood Distribution (Call Counts) ---
dispatch_neighborhood
-                                    5143
ALASKA JUNCTION                      5792
ALKI                                 3431
BALLARD NORTH                        4438
BALLARD SOUTH                       10170
BELLTOWN                             7683
BITTERLAKE                           7245
BRIGHTON/DUNLAP                      3713
CAPITOL HILL                        24465
CENTRAL AREA/SQUIRE PARK             8016
CHINATOWN/INTERNATIONAL DISTRICT    15218
CLAREMONT/RAINIER VISTA              2335
COLUMBIA CITY                        1633
COMMERCIAL DUWAMISH                   726
COMMERCIAL HARBOR ISLAND              127
DOWNTOWN COMMERCIAL                 20095
EASTLAKE - EAST                       400
EASTLAKE - WEST                      1351
FAUNTLEROY SW                         873
FIRST HILL                          11682
FREMONT                              4179
GENESEE